# Marketplace Growth & Seller Analytics

Python / Google Colab analysis for a simulated two-sided marketplace.

**Analysis period:** July 2025 through June 2026

This notebook complements the BigQuery SQL analysis by validating KPI results, analyzing trends, reshaping cohort data, and creating portfolio-ready visualizations.


## 1. Setup

This notebook is designed for Google Colab and uses BigQuery as the data source.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_id = "marketplace-analytics-project"
client = bigquery.Client(project=project_id)

print("Connected to BigQuery!")


## 2. Monthly GMV Trend

This analysis measures month-by-month GMV growth and identifies the strongest and weakest month-over-month growth periods.


In [ ]:
query = """
SELECT
  DATE_TRUNC(o.order_date, MONTH) AS month,
  ROUND(SUM(oi.sale_price * oi.quantity), 2) AS monthly_gmv
FROM `marketplace-analytics-project.marketplace_clean.orders_clean` o
JOIN `marketplace-analytics-project.marketplace_clean.order_items_clean` oi
  ON o.order_id = oi.order_id
WHERE o.order_status = 'completed'
GROUP BY month
ORDER BY month
"""

monthly_gmv = client.query(query).to_dataframe()
monthly_gmv["month"] = pd.to_datetime(monthly_gmv["month"])
monthly_gmv["monthly_gmv"] = monthly_gmv["monthly_gmv"].astype(float)

monthly_gmv


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(monthly_gmv["month"], monthly_gmv["monthly_gmv"], marker="o")
plt.title("Monthly Marketplace GMV")
plt.xlabel("Month")
plt.ylabel("GMV ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
monthly_gmv["mom_growth_pct"] = (
    monthly_gmv["monthly_gmv"].pct_change().mul(100).round(2)
)

best_growth_month = monthly_gmv.loc[
    monthly_gmv["mom_growth_pct"].idxmax()
]
lowest_growth_month = monthly_gmv.loc[
    monthly_gmv["mom_growth_pct"].idxmin()
]

first_gmv = monthly_gmv.iloc[0]["monthly_gmv"]
last_gmv = monthly_gmv.iloc[-1]["monthly_gmv"]
overall_growth_pct = ((last_gmv - first_gmv) / first_gmv) * 100

print(f"Strongest MoM growth: {best_growth_month['month']:%b %Y} ({best_growth_month['mom_growth_pct']:.2f}%)")
print(f"Weakest MoM growth: {lowest_growth_month['month']:%b %Y} ({lowest_growth_month['mom_growth_pct']:.2f}%)")
print(f"Overall GMV growth: {overall_growth_pct:.2f}%")


**Key finding:** Monthly GMV increased from about **$44K to $107K**, representing approximately **141% growth** across the analysis period.


## 3. Category GMV

This section compares marketplace revenue contribution across product categories.


In [ ]:
category_query = """
SELECT
  l.category,
  ROUND(SUM(oi.sale_price * oi.quantity), 2) AS category_gmv
FROM `marketplace-analytics-project.marketplace_clean.orders_clean` o
JOIN `marketplace-analytics-project.marketplace_clean.order_items_clean` oi
  ON o.order_id = oi.order_id
JOIN `marketplace-analytics-project.marketplace_clean.listings_clean` l
  ON oi.listing_id = l.listing_id
WHERE o.order_status = 'completed'
GROUP BY l.category
ORDER BY category_gmv DESC
"""

category_gmv = client.query(category_query).to_dataframe()
category_gmv["category_gmv"] = category_gmv["category_gmv"].astype(float)

total_gmv = category_gmv["category_gmv"].sum()
category_gmv["gmv_share_pct"] = (
    category_gmv["category_gmv"] / total_gmv * 100
).round(2)

category_gmv


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(category_gmv["category"], category_gmv["category_gmv"])
plt.title("GMV by Product Category")
plt.xlabel("Category")
plt.ylabel("GMV ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Key finding:** Electronics generated the most GMV, while Trading Cards and Collectibles later emerged as stronger categories for repeat purchase behavior.


## 4. Buyer Cohort Retention

Cohort retention measures whether buyers return in later months after their first observed purchase.


In [ ]:
retention_query = """
WITH completed_orders AS (
  SELECT *
  FROM `marketplace-analytics-project.marketplace_clean.orders_clean`
  WHERE order_status = 'completed'
),
first_purchase AS (
  SELECT
    buyer_id,
    DATE_TRUNC(MIN(order_date), MONTH) AS cohort_month
  FROM completed_orders
  GROUP BY buyer_id
),
buyer_activity AS (
  SELECT DISTINCT
    buyer_id,
    DATE_TRUNC(order_date, MONTH) AS activity_month
  FROM completed_orders
),
cohort_activity AS (
  SELECT
    f.buyer_id,
    f.cohort_month,
    a.activity_month,
    DATE_DIFF(a.activity_month, f.cohort_month, MONTH) AS month_number
  FROM first_purchase f
  JOIN buyer_activity a
    ON f.buyer_id = a.buyer_id
  WHERE a.activity_month >= f.cohort_month
),
cohort_sizes AS (
  SELECT
    cohort_month,
    COUNT(DISTINCT buyer_id) AS cohort_size
  FROM first_purchase
  GROUP BY cohort_month
)
SELECT
  c.cohort_month,
  c.month_number,
  s.cohort_size,
  COUNT(DISTINCT c.buyer_id) AS retained_buyers,
  ROUND(
    SAFE_DIVIDE(COUNT(DISTINCT c.buyer_id), s.cohort_size) * 100,
    2
  ) AS retention_rate
FROM cohort_activity c
JOIN cohort_sizes s
  USING (cohort_month)
GROUP BY c.cohort_month, c.month_number, s.cohort_size
ORDER BY c.cohort_month, c.month_number
"""

retention = client.query(retention_query).to_dataframe()
retention["cohort_month"] = pd.to_datetime(retention["cohort_month"])
retention["retention_rate"] = retention["retention_rate"].astype(float)

retention.head()


In [ ]:
retention_matrix = retention.pivot(
    index="cohort_month",
    columns="month_number",
    values="retention_rate"
)

retention_matrix.index = pd.to_datetime(retention_matrix.index)

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(retention_matrix.values, aspect="auto")

ax.set_xticks(range(len(retention_matrix.columns)))
ax.set_xticklabels(retention_matrix.columns)

ax.set_yticks(range(len(retention_matrix.index)))
ax.set_yticklabels(retention_matrix.index.strftime("%b %Y"))

ax.set_xlabel("Months Since First Purchase")
ax.set_ylabel("Buyer Cohort")
ax.set_title("Buyer Cohort Retention")

for i in range(retention_matrix.shape[0]):
    for j in range(retention_matrix.shape[1]):
        value = retention_matrix.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:.1f}%", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
month_1_retention = retention[
    retention["month_number"] == 1
].copy()

plt.figure(figsize=(10, 6))
plt.plot(
    month_1_retention["cohort_month"],
    month_1_retention["retention_rate"],
    marker="o"
)
plt.title("Month-1 Retention by Buyer Cohort")
plt.xlabel("Cohort Month")
plt.ylabel("Retention Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Key finding:** Month-1 retention declined substantially across newer cohorts, identifying early buyer retention as a major marketplace risk.


## 5. New vs Returning Buyer GMV

This analysis compares how much monthly GMV comes from buyers making their first observed purchase versus buyers who had purchased previously.


In [ ]:
buyer_mix_query = """
WITH completed_orders AS (
  SELECT *
  FROM `marketplace-analytics-project.marketplace_clean.orders_clean`
  WHERE order_status = 'completed'
),
first_purchase AS (
  SELECT
    buyer_id,
    DATE_TRUNC(MIN(order_date), MONTH) AS first_purchase_month
  FROM completed_orders
  GROUP BY buyer_id
)
SELECT
  DATE_TRUNC(o.order_date, MONTH) AS month,
  ROUND(SUM(CASE
    WHEN DATE_TRUNC(o.order_date, MONTH) = f.first_purchase_month
      THEN o.order_total ELSE 0 END), 2) AS new_buyer_gmv,
  ROUND(SUM(CASE
    WHEN DATE_TRUNC(o.order_date, MONTH) > f.first_purchase_month
      THEN o.order_total ELSE 0 END), 2) AS returning_buyer_gmv
FROM completed_orders o
JOIN first_purchase f
  ON o.buyer_id = f.buyer_id
GROUP BY month
ORDER BY month
"""

buyer_mix = client.query(buyer_mix_query).to_dataframe()
buyer_mix["month"] = pd.to_datetime(buyer_mix["month"])
buyer_mix["new_buyer_gmv"] = buyer_mix["new_buyer_gmv"].astype(float)
buyer_mix["returning_buyer_gmv"] = buyer_mix["returning_buyer_gmv"].astype(float)

buyer_mix


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    buyer_mix["month"],
    buyer_mix["new_buyer_gmv"],
    marker="o",
    label="New Buyer GMV"
)
plt.plot(
    buyer_mix["month"],
    buyer_mix["returning_buyer_gmv"],
    marker="o",
    label="Returning Buyer GMV"
)

plt.title("Monthly GMV: New vs Returning Buyers")
plt.xlabel("Month")
plt.ylabel("GMV ($)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Key finding:** Returning buyers became the primary source of monthly GMV and generated about **69% of June 2026 GMV**.


## 6. Top Sellers and Concentration Risk

This analysis validates how much marketplace GMV is concentrated among the ten highest-GMV sellers.


In [ ]:
top_sellers_query = """
SELECT
  seller_id,
  ROUND(SUM(order_total), 2) AS seller_gmv
FROM `marketplace-analytics-project.marketplace_clean.orders_clean`
WHERE order_status = 'completed'
GROUP BY seller_id
ORDER BY seller_gmv DESC
LIMIT 10
"""

top_sellers = client.query(top_sellers_query).to_dataframe()
top_sellers["seller_gmv"] = top_sellers["seller_gmv"].astype(float)
top_sellers["seller_id"] = top_sellers["seller_id"].astype(str)

top_sellers


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(top_sellers["seller_id"], top_sellers["seller_gmv"])
plt.title("Top 10 Sellers by GMV")
plt.xlabel("Seller ID")
plt.ylabel("GMV ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

top_10_gmv = top_sellers["seller_gmv"].sum()
marketplace_gmv = monthly_gmv["monthly_gmv"].sum()
top_10_share = top_10_gmv / marketplace_gmv * 100

print(f"Top 10 Seller GMV: ${top_10_gmv:,.2f}")
print(f"Marketplace GMV: ${marketplace_gmv:,.2f}")
print(f"Top 10 GMV Share: {top_10_share:.2f}%")


**Key finding:** The Top 10 sellers generated **47.49% of total marketplace GMV**, creating meaningful seller concentration risk.


## 7. Executive KPI Validation

The final check compares the six executive KPIs calculated in BigQuery with the values used for reporting.


In [ ]:
kpi_query = """
SELECT *
FROM `marketplace-analytics-project.marketplace_clean.executive_kpis`
"""

kpi_summary = client.query(kpi_query).to_dataframe()
kpi_summary


In [ ]:
kpi_display = pd.DataFrame({
    "KPI": [
        "Total GMV",
        "Active Buyers",
        "Active Sellers",
        "Repeat Purchase Rate",
        "Seller Sell-Through Rate",
        "Top-10 Seller GMV Share"
    ],
    "Value": [
        f"${float(kpi_summary.loc[0, 'total_gmv']):,.2f}",
        f"{int(kpi_summary.loc[0, 'active_buyers']):,}",
        f"{int(kpi_summary.loc[0, 'active_sellers']):,}",
        f"{float(kpi_summary.loc[0, 'repeat_purchase_rate']):.2f}%",
        f"{float(kpi_summary.loc[0, 'sell_through_rate']):.2f}%",
        f"{float(kpi_summary.loc[0, 'top_10_gmv_share']):.2f}%"
    ]
})

kpi_display


## Final Takeaway

The marketplace showed strong topline growth, but the deeper analysis revealed two important risks:

1. **Early buyer retention weakened materially across newer cohorts.**
2. **Nearly half of marketplace GMV was concentrated among only ten sellers.**

The business opportunity is therefore not simply to acquire more users, but to convert more first-time buyers into repeat customers and develop a broader base of successful sellers.
